In [ ]:
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format
pd.options.display.width = 200
df = pd.read_csv('paket_a_transaksi_pos_2024_2025.csv')
print(df.shape)
print(df.dtypes)
df.head()

(5030, 9)
id_transaksi       str
tanggal            str
kode_gerai         str
kode_kasir         str
kode_produk        str
jumlah           int64
harga_satuan       str
diskon_persen    int64
status             str
dtype: object


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status
0,TRX0241065,2025-06-02,G09,K011,RMT1402,4,13000,0,SELESAI
1,TRX0240510,2024-12-28,G09,K015,SGR1205,1,21000,0,SELESAI
2,TRX0244483,2024-09-04,G11,K029,RMT1407,3,"7500,00",0,SELESAI
3,TRX0242532,2025-07-30,G05,K001,MKN1103,2,6500,0,SELESAI
4,TRX0243509,26/03/2025,G05,K016,RMT1407,1,6000,0,SELESAI


# Task 1 — Indexing & Selection (Pembersihan Anomali & Imputasi)

### 1.1 Pembersihan format harga

In [ ]:
print('Sebelum:', df.harga_satuan.dtype)
print('Contoh nilai berkoma :', df.harga_satuan[df.harga_satuan.str.contains(',')].head(3).tolist())
print('Contoh nilai bertitik:', df.harga_satuan[df.harga_satuan.str.contains(r'\.')].head(3).tolist())

df['harga_satuan'] = (df['harga_satuan']
                      .str.replace('.', '', regex=False)    # buang titik pemisah ribuan ('28.000' -> '28000')
                      .str.replace(',', '.', regex=False)   # koma desimal -> titik ('7500,00' -> '7500.00')
                      .astype(float))
print('Sesudah:', df.harga_satuan.dtype)
df.harga_satuan.describe()

Sebelum: str
Contoh nilai berkoma : ['7500,00', '11000,00', '27000,00']
Contoh nilai bertitik: ['13.000', '28.000', '11.000']
Sesudah: float64


count    5,030.00
mean    22,185.09
std     15,205.70
min      4,000.00
25%     11,000.00
50%     16,000.00
75%     32,000.00
max     54,000.00

### 1.2 Kuantitas negatif → positif

In [ ]:
print('Jumlah negatif sebelum:', (df.jumlah < 0).sum())
df.loc[df['jumlah'] < 0, 'jumlah'] = df.loc[df['jumlah'] < 0, 'jumlah'].abs()
print('Jumlah negatif sesudah :', (df.jumlah < 0).sum())

Jumlah negatif sebelum: 41
Jumlah negatif sesudah : 0


### 1.3 Imputasi kode_kasir kosong → K_UNKNOWN

In [ ]:
print('NaN kode_kasir sebelum:', df.kode_kasir.isna().sum())
df.loc[df['kode_kasir'].isna(), 'kode_kasir'] = 'K_UNKNOWN'
print('NaN kode_kasir sesudah :', df.kode_kasir.isna().sum())
print('Total missing value tersisa:', df.isna().sum().sum())

NaN kode_kasir sebelum: 261
NaN kode_kasir sesudah : 0
Total missing value tersisa: 0


### 1.4 Slicing posisi dengan `.iloc[]` (5 baris pertama, 4 kolom)

In [ ]:
kolom = [df.columns.get_loc(c) for c in ['id_transaksi', 'tanggal', 'kode_gerai', 'jumlah']]
df.iloc[:5, kolom]

,id_transaksi,tanggal,kode_gerai,jumlah
0,TRX0241065,2025-06-02,G09,4
1,TRX0240510,2024-12-28,G09,1
2,TRX0244483,2024-09-04,G11,3
3,TRX0242532,2025-07-30,G05,2
4,TRX0243509,26/03/2025,G05,1


# Task 2 — Operasi Aritmatika (Feature Engineering)

In [ ]:
df['Total_Kotor'] = df['harga_satuan'] * df['jumlah']
df['Nominal_Diskon'] = df['Total_Kotor'] * (df['diskon_persen'] / 100)
df['Total_Bersih'] = (df['Total_Kotor'] - df['Nominal_Diskon']) * 1.11
df[['id_transaksi', 'harga_satuan', 'jumlah', 'diskon_persen', 'Total_Kotor', 'Nominal_Diskon', 'Total_Bersih']].head()

,id_transaksi,harga_satuan,jumlah,diskon_persen,Total_Kotor,Nominal_Diskon,Total_Bersih
0,TRX0241065,"13,000.00",4,0,"52,000.00",0.00,"57,720.00"
1,TRX0240510,"21,000.00",1,0,"21,000.00",0.00,"23,310.00"
2,TRX0244483,"7,500.00",3,0,"22,500.00",0.00,"24,975.00"
3,TRX0242532,"6,500.00",2,0,"13,000.00",0.00,"14,430.00"
4,TRX0243509,"6,000.00",1,0,"6,000.00",0.00,"6,660.00"


# Task 3 — Filtering (Penyaringan Data)

### 3.1 Filter multi-kondisi: SELESAI, gerai G05 atau G09, Total_Bersih > Rp100.000

In [ ]:
f1 = df[(df['status'] == 'SELESAI')
        & (df['kode_gerai'].isin(['G05', 'G09']))
        & (df['Total_Bersih'] > 100_000)]
print('Jumlah baris yang memenuhi kriteria:', len(f1))
f1.head(10)

Jumlah baris yang memenuhi kriteria: 140


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
7,TRX0241251,2024-10-09,G09,K035,SGR1204,4,"47,500.00",10,SELESAI,"190,000.00","19,000.00","189,810.00"
24,TRX0244253,2024-07-18,G09,K026,BYI1502,4,"30,500.00",0,SELESAI,"122,000.00",0.00,"135,420.00"
25,TRX0244999,2024-08-09,G05,K015,MIN1003,4,"28,500.00",0,SELESAI,"114,000.00",0.00,"126,540.00"
90,TRX0240898,2025-12-09,G05,K010,BYI1509,3,"46,500.00",0,SELESAI,"139,500.00",0.00,"154,845.00"
206,TRX0241868,2025-07-01,G05,K009,SGR1203,6,"20,500.00",0,SELESAI,"123,000.00",0.00,"136,530.00"
213,TRX0244651,2025-04-15,G09,K010,PRW1303,2,"46,000.00",0,SELESAI,"92,000.00",0.00,"102,120.00"
216,TRX0241198,2024-06-16,G09,K020,BYI1502,6,"28,000.00",0,SELESAI,"168,000.00",0.00,"186,480.00"
220,TRX0241165,2025-09-24,G09,K023,SGR1208,6,"27,500.00",0,SELESAI,"165,000.00",0.00,"183,150.00"
373,TRX0240601,2025-04-28,G05,K040,SGR1204,12,"50,000.00",0,SELESAI,"600,000.00",0.00,"666,000.00"
410,TRX0241526,2024-12-31,G05,K026,BYI1501,6,"18,500.00",0,SELESAI,"111,000.00",0.00,"123,210.00"


### 3.2 Filter dengan .query(): gerai G10, jumlah minimal 5 unit

In [ ]:
f2 = df.query("kode_gerai == 'G10' and jumlah >= 5")
print('Jumlah baris:', len(f2))
f2

Jumlah baris: 39


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
83,TRX0243880,2024-08-04,G10,K002,MKN1101,12,"45,500.00",0,SELESAI,"546,000.00",0.00,"606,060.00"
166,TRX0241592,2024-05-11,G10,K033,SGR1202,6,"13,000.00",0,SELESAI,"78,000.00",0.00,"86,580.00"
351,TRX0242686,2024-02-03,G10,K016,BYI1502,12,"26,000.00",0,SELESAI,"312,000.00",0.00,"346,320.00"
556,TRX0241396,2025-12-16,G10,K009,RMT1406,12,"5,500.00",5,SELESAI,"66,000.00","3,300.00","69,597.00"
879,TRX0243559,2025-09-30,G10,K016,SGR1206,6,"47,000.00",0,SELESAI,"282,000.00",0.00,"313,020.00"
964,TRX0244939,2025-07-08,G10,K009,SGR1200,6,"52,000.00",10,SELESAI,"312,000.00","31,200.00","311,688.00"
989,TRX0243076,2025-02-18,G10,K003,MKN1100,6,"18,000.00",0,SELESAI,"108,000.00",0.00,"119,880.00"
1023,TRX0242437,2025-09-18,G10,K030,MKN1104,6,"6,000.00",0,SELESAI,"36,000.00",0.00,"39,960.00"
1260,TRX0240537,2025-05-09,G10,K027,BYI1509,6,"54,000.00",0,SELESAI,"324,000.00",0.00,"359,640.00"
1296,TRX0240698,2024-01-24,G10,K022,MKN1109,6,"29,000.00",0,SELESAI,"174,000.00",0.00,"193,140.00"


# Task 4 — Sorting (Pengurutan Data)

### 4.1 Urut `kode_gerai` A–Z, lalu `Total_Bersih` tertinggi → terendah

In [ ]:
df_sorted = df.sort_values(by=['kode_gerai', 'Total_Bersih'], ascending=[True, False])
df_sorted[['id_transaksi', 'kode_gerai', 'Total_Bersih']].head(10)

,id_transaksi,kode_gerai,Total_Bersih
244,TRX0243257,G01,"42,020,160.00"
2956,TRX0241367,G01,"672,660.00"
3588,TRX0240608,G01,"626,040.00"
2937,TRX0240631,G01,"559,440.00"
5,TRX0240658,G01,"545,454.00"
3889,TRX0241537,G01,"359,640.00"
2179,TRX0244635,G01,"349,650.00"
4270,TRX0240428,G01,"323,010.00"
695,TRX0244706,G01,"313,020.00"
4404,TRX0242179,G01,"313,020.00"


### 4.2 Top-5 transaksi dengan Total_Bersih tertinggi (.nlargest())

In [ ]:
top5 = df.nlargest(5, 'Total_Bersih')
top5[['id_transaksi', 'tanggal', 'kode_gerai', 'status', 'jumlah', 'harga_satuan', 'diskon_persen', 'Total_Bersih']]

,id_transaksi,tanggal,kode_gerai,status,jumlah,harga_satuan,diskon_persen,Total_Bersih
994,TRX0241002,2024-03-18,G11,SELESAI,2093,"48,000.00",10,"100,363,536.00"
4985,TRX0244221,2024-12-09,G09,SELESAI,1301,"51,500.00",10,"66,934,498.50"
467,TRX0243696,2024-02-27,G06,SELESAI,892,"54,000.00",0,"53,466,480.00"
3393,TRX0241029,2025-05-12,G07,SELESAI,1983,"23,000.00",10,"45,563,391.00"
244,TRX0243257,2024-11-24,G01,SELESAI,1456,"32,500.00",20,"42,020,160.00"


> Catatan: Top-5 didominasi baris dengan jumlah ribuan (mis. 2.375 unit) yang tampak seperti kesalahan input. Task ini tidak meminta membuangnya, sehingga tetap dihitung apa adanya.

# Task 5 — Grouping & Agregasi (Pelaporan Kinerja Gerai)

### 5.1 laporan_gerai (hanya status SELESAI)

In [ ]:
selesai = df[df['status'] == 'SELESAI']
laporan_gerai = (selesai.groupby('kode_gerai')
                 .agg(Jumlah_Transaksi=('id_transaksi', 'count'),
                      Total_Unit_Terjual=('jumlah', 'sum'),
                      Total_Omzet_Bersih=('Total_Bersih', 'sum'),
                      Rata_Rata_Transaksi=('Total_Bersih', 'mean'))
                 .sort_values('Total_Omzet_Bersih', ascending=False)
                 .round(2))
laporan_gerai

,Jumlah_Transaksi,Total_Unit_Terjual,Total_Omzet_Bersih,Rata_Rata_Transaksi
kode_gerai,,,,
G11,398,3170,"125,638,152.75","315,673.75"
G09,379,3027,"101,157,630.00","266,906.68"
G06,383,3103,"91,228,624.50","238,194.84"
G07,384,4284,"80,559,748.50","209,791.01"
G01,450,2635,"70,967,378.25","157,705.28"
G03,372,3274,"59,991,226.50","161,266.74"
G10,401,1742,"58,964,892.75","147,044.62"
G08,408,1700,"35,630,084.25","87,328.64"
G12,406,1010,"24,699,220.50","60,835.52"


### 5.2 Kontribusi_Gerai_Persen dengan .groupby().transform()
(Total_Bersih / Total_Omzet_Gerai) × 100. Total omzet gerai dihitung dari transaksi SELESAI (sama dengan basis laporan_gerai); baris berstatus lain tetap dihitung kontribusinya terhadap total tersebut.

In [ ]:
total_gerai = df['Total_Bersih'].where(df['status'] == 'SELESAI').groupby(df['kode_gerai']).transform('sum')
df['Kontribusi_Gerai_Persen'] = df['Total_Bersih'] / total_gerai * 100
print('Cek: jumlah kontribusi SELESAI per gerai harus 100')
print(df[df.status == 'SELESAI'].groupby('kode_gerai')['Kontribusi_Gerai_Persen'].sum().round(2).to_dict())
df[['id_transaksi', 'kode_gerai', 'status', 'Total_Bersih', 'Kontribusi_Gerai_Persen']].head(10)

Cek: jumlah kontribusi SELESAI per gerai harus 100
{'G01': 100.0, 'G02': 100.0, 'G03': 100.0, 'G04': 100.0, 'G05': 100.0, 'G06': 100.0, 'G07': 100.0, 'G08': 100.0, 'G09': 100.0, 'G10': 100.0, 'G11': 100.0, 'G12': 100.0}


,id_transaksi,kode_gerai,status,Total_Bersih,Kontribusi_Gerai_Persen
0,TRX0241065,G09,SELESAI,"57,720.00",0.06
1,TRX0240510,G09,SELESAI,"23,310.00",0.02
2,TRX0244483,G11,SELESAI,"24,975.00",0.02
3,TRX0242532,G05,SELESAI,"14,430.00",0.06
4,TRX0243509,G05,SELESAI,"6,660.00",0.03
5,TRX0240658,G01,SELESAI,"545,454.00",0.77
6,TRX0244581,G10,SELESAI,"19,980.00",0.03
7,TRX0241251,G09,SELESAI,"189,810.00",0.19
8,TRX0244274,G07,SELESAI,"36,630.00",0.05
9,TRX0243163,G08,BATAL,"14,430.00",0.04
